# Build Panel Dataset from Raw Sources

Notebook ini menggabungkan data mentah multi-sumber (`BKPM`, `IPM`, `PDRB`, `KEMISKINAN`, `TPT`, `AKSES LISTRIK`) menjadi:

1. **Panel tahunan provinsi** (`provinsi`, `year`, indikator)
2. **Final average dataset** siap upload backend (`provinsi` + 7 indikator)

## 1) Setup

In [ ]:
from pathlib import Path
from datetime import datetime
import re
    
import numpy as np
import pandas as pd

print('pandas:', pd.__version__)

cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    WORK_DIR = cwd
elif (cwd.parent / 'data').exists():
    WORK_DIR = cwd.parent
else:
    raise RuntimeError('Folder data tidak ditemukan. Jalankan notebook dari folder investra-gov-apps-data-prep atau subfolder notebooks.')

RAW_DIR = WORK_DIR / 'data' / 'raw'
INTERIM_DIR = WORK_DIR / 'data' / 'interim'
FINAL_DIR = WORK_DIR / 'data' / 'final'

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print('RAW_DIR   :', RAW_DIR)
print('INTERIM   :', INTERIM_DIR)
print('FINAL     :', FINAL_DIR)

## 2) Konfigurasi dan Utilitas

In [ ]:
CANONICAL_PROVINCES = [
    'ACEH','SUMATERA UTARA','SUMATERA BARAT','RIAU','JAMBI','SUMATERA SELATAN','BENGKULU','LAMPUNG',
    'KEPULAUAN BANGKA BELITUNG','KEPULAUAN RIAU','DKI JAKARTA','JAWA BARAT','JAWA TENGAH','DI YOGYAKARTA','JAWA TIMUR',
    'BANTEN','BALI','NUSA TENGGARA BARAT','NUSA TENGGARA TIMUR','KALIMANTAN BARAT','KALIMANTAN TENGAH',
    'KALIMANTAN SELATAN','KALIMANTAN TIMUR','KALIMANTAN UTARA','SULAWESI UTARA','SULAWESI TENGAH',
    'SULAWESI SELATAN','SULAWESI TENGGARA','GORONTALO','SULAWESI BARAT','MALUKU','MALUKU UTARA',
    'PAPUA BARAT','PAPUA BARAT DAYA','PAPUA','PAPUA SELATAN','PAPUA TENGAH','PAPUA PEGUNUNGAN'
]
CANONICAL_SET = set(CANONICAL_PROVINCES)

PROVINCE_ALIASES = {
    'KEP. BANGKA BELITUNG': 'KEPULAUAN BANGKA BELITUNG',
    'KEP. RIAU': 'KEPULAUAN RIAU',
    'DAERAH KHUSUS IBUKOTA JAKARTA': 'DKI JAKARTA',
    'DAERAH ISTIMEWA YOGYAKARTA': 'DI YOGYAKARTA',
    'D.I. YOGYAKARTA': 'DI YOGYAKARTA',
    'KOTA ADM. JAKARTA': 'DKI JAKARTA',
}

def normalize_text(value: str) -> str:
    text = str(value).strip().upper()
    text = re.sub(r'\s+', ' ', text)
    return text

def normalize_province(value: str) -> str | None:
    p = normalize_text(value)
    if not p or p in {'NAN', 'INDONESIA', '38 PROVINSI', 'PROVINSI'}:
        return None
    p = PROVINCE_ALIASES.get(p, p)
    return p if p in CANONICAL_SET else None

def extract_year(text: str) -> int:
    m = re.search(r'(20\d{2})', str(text))
    if not m:
        raise ValueError(f'Tahun tidak ditemukan: {text}')
    return int(m.group(1))

def to_numeric_list(values) -> list[float]:
    ser = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    return ser.astype(float).tolist()

## 3) Loader BKPM (PMDN/PMA per Provinsi-Tahun)

In [ ]:
def load_bkpm_annual(raw_root: Path) -> pd.DataFrame:
    csv_files = sorted((raw_root / 'BKPM').rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError('File BKPM .csv tidak ditemukan')

    parts = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, low_memory=False)
        except Exception:
            continue

        required_cols = {'periode', 'status_penanaman_modal', 'provinsi', 'investasi_rp_juta'}
        if not required_cols.issubset(set(df.columns)):
            continue

        temp = df[['periode', 'status_penanaman_modal', 'provinsi', 'investasi_rp_juta']].copy()
        temp['year'] = temp['periode'].astype(str).map(extract_year)
        temp['provinsi'] = temp['provinsi'].map(normalize_province)
        temp = temp[temp['provinsi'].notna()].copy()
        temp['status'] = temp['status_penanaman_modal'].astype(str).str.upper().str.strip()
        temp['investasi_rp_juta'] = pd.to_numeric(temp['investasi_rp_juta'], errors='coerce').fillna(0.0)

        grp = (
            temp.groupby(['year', 'provinsi', 'status'], as_index=False)['investasi_rp_juta']
            .sum()
        )
        parts.append(grp)

    if not parts:
        raise RuntimeError('Tidak ada data BKPM yang berhasil diparse')

    merged = pd.concat(parts, ignore_index=True)
    merged = (
        merged.groupby(['year', 'provinsi', 'status'], as_index=False)['investasi_rp_juta']
        .sum()
    )

    pivot = merged.pivot_table(
        index=['year', 'provinsi'],
        columns='status',
        values='investasi_rp_juta',
        aggfunc='sum',
        fill_value=0.0,
    ).reset_index()

    pmdn_juta = pivot.filter(regex='^PMDN$').sum(axis=1)
    pma_juta = pivot.filter(regex='^PMA$').sum(axis=1)

    out = pivot[['year', 'provinsi']].copy()
    out['pmdn_rp'] = pmdn_juta * 1_000_000.0
    out['fdi_rp'] = pma_juta * 1_000_000.0
    return out

bkpm_df = load_bkpm_annual(RAW_DIR)
print('BKPM rows:', len(bkpm_df))
display(bkpm_df.head())

## 4) Loader Indikator BPS

In [ ]:
def load_indicator_simple(folder: Path, value_col: str, strategy: str = 'first') -> pd.DataFrame:
    rows = []
    for f in sorted(folder.glob('*.csv')):
        year = extract_year(f.name)
        raw = pd.read_csv(f, header=None)
        for rec in raw.itertuples(index=False):
            prov = normalize_province(rec[0])
            if prov is None:
                continue
            nums = to_numeric_list(rec[1:])
            if not nums:
                continue
            if strategy == 'first':
                val = nums[0]
            elif strategy == 'mean_all':
                val = float(np.mean(nums))
            elif strategy == 'mean_last_two':
                val = float(np.mean(nums[-2:])) if len(nums) >= 2 else float(np.mean(nums))
            else:
                raise ValueError(f'strategy tidak dikenal: {strategy}')
            rows.append({'year': year, 'provinsi': prov, value_col: val})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'Data {value_col} kosong dari folder: {folder}')
    return df.groupby(['year', 'provinsi'], as_index=False)[value_col].mean()

akses_df = load_indicator_simple(RAW_DIR / 'AKSES LISTRIK', 'akses_listrik', strategy='first')
ipm_df = load_indicator_simple(RAW_DIR / 'IPM', 'ipm', strategy='first')
kemiskinan_df = load_indicator_simple(RAW_DIR / 'KEMISKINAN', 'kemiskinan', strategy='mean_last_two')
tpt_df = load_indicator_simple(RAW_DIR / 'TPT', 'tpt', strategy='mean_all')
pdrb_df = load_indicator_simple(RAW_DIR / 'PDRB', 'pdrb_per_kapita', strategy='first')
pdrb_df['pdrb_per_kapita'] = pdrb_df['pdrb_per_kapita'] * 1_000.0  # source dalam ribu rupiah

print('akses:', akses_df.shape, '| ipm:', ipm_df.shape, '| miskin:', kemiskinan_df.shape, '| tpt:', tpt_df.shape, '| pdrb:', pdrb_df.shape)

## 5) Build Panel Tahunan

In [ ]:
all_years = sorted(set(bkpm_df['year']))
full_grid = pd.MultiIndex.from_product([all_years, CANONICAL_PROVINCES], names=['year', 'provinsi']).to_frame(index=False)

panel = full_grid.copy()
panel = panel.merge(bkpm_df, on=['year', 'provinsi'], how='left')
panel = panel.merge(pdrb_df, on=['year', 'provinsi'], how='left')
panel = panel.merge(ipm_df, on=['year', 'provinsi'], how='left')
panel = panel.merge(kemiskinan_df, on=['year', 'provinsi'], how='left')
panel = panel.merge(akses_df, on=['year', 'provinsi'], how='left')
panel = panel.merge(tpt_df, on=['year', 'provinsi'], how='left')

# Missing investasi = 0 (tidak ada realisasi)
panel['pmdn_rp'] = panel['pmdn_rp'].fillna(0.0)
panel['fdi_rp'] = panel['fdi_rp'].fillna(0.0)

# Untuk indikator lain, interpolasi per provinsi jika ada missing
for col in ['pdrb_per_kapita', 'ipm', 'kemiskinan', 'akses_listrik', 'tpt']:
    panel[col] = (
        panel.sort_values('year')
        .groupby('provinsi')[col]
        .transform(lambda s: s.interpolate(limit_direction='both'))
    )

# Drop jika masih ada yang kosong
panel = panel.dropna(subset=['pdrb_per_kapita', 'ipm', 'kemiskinan', 'akses_listrik', 'tpt']).copy()

# Boundaries sesuai rule backend
for col in ['ipm', 'kemiskinan', 'akses_listrik', 'tpt']:
    panel[col] = panel[col].clip(lower=0, upper=100)
for col in ['pmdn_rp', 'fdi_rp', 'pdrb_per_kapita']:
    panel[col] = panel[col].clip(lower=0)

panel = panel.sort_values(['year', 'provinsi']).reset_index(drop=True)
print('Panel shape:', panel.shape)
display(panel.head())
display(panel.tail())

## 6) Validasi Coverage

In [ ]:
coverage = panel.groupby('year')['provinsi'].nunique().reset_index(name='province_count')
display(coverage)

if (coverage['province_count'] < len(CANONICAL_PROVINCES)).any():
    print('WARNING: Ada tahun dengan jumlah provinsi < 38')
else:
    print('OK: Semua tahun lengkap 38 provinsi')

## 7) Export Panel + Final CSV Siap Upload

In [ ]:
REQUIRED_UPLOAD_COLS = ['provinsi', 'pmdn_rp', 'fdi_rp', 'pdrb_per_kapita', 'ipm', 'kemiskinan', 'akses_listrik', 'tpt']

AGG_START_YEAR = 2022
AGG_END_YEAR = 2024
EXPORT_SINGLE_YEAR = 2024

ts = datetime.now().strftime('%Y%m%d_%H%M%S')

# Panel export
panel_path = INTERIM_DIR / f'investra_panel_{panel.year.min()}_{panel.year.max()}_{ts}.csv'
panel.to_csv(panel_path, index=False)

# Average export (mis. 2022-2024) untuk backend yang pakai satu snapshot
window = panel[(panel['year'] >= AGG_START_YEAR) & (panel['year'] <= AGG_END_YEAR)].copy()
final_avg = window.groupby('provinsi', as_index=False)[REQUIRED_UPLOAD_COLS[1:]].mean()
final_avg = final_avg[REQUIRED_UPLOAD_COLS].sort_values('provinsi').reset_index(drop=True)
final_avg_path = FINAL_DIR / f'investra_final_avg_{AGG_START_YEAR}_{AGG_END_YEAR}_{ts}.csv'
final_avg.to_csv(final_avg_path, index=False)

# Single year export (mis. 2024)
final_year = panel[panel['year'] == EXPORT_SINGLE_YEAR][REQUIRED_UPLOAD_COLS].copy()
final_year = final_year.sort_values('provinsi').reset_index(drop=True)
final_year_path = FINAL_DIR / f'investra_final_{EXPORT_SINGLE_YEAR}_{ts}.csv'
final_year.to_csv(final_year_path, index=False)

print('Saved panel :', panel_path)
print('Saved final avg :', final_avg_path)
print('Saved final year:', final_year_path)
print('\nPreview final avg:')
display(final_avg.head())